# Import libraries

In [ ]:
import numpy as np
import pandas as pd
# import cudf as pd
# import cupy as cp
from scipy import stats

import concurrent.futures

import seaborn as sns

# pd.set_option('max_rows', 300)
# pd.set_option('max_columns', 300)

from tqdm.notebook import tqdm

import optiver_feature_creation as opt
import optiver_feature_creation2 as opt2
import optiver_feature_creation3 as opt3

import gc

In [ ]:
! ls -hlt ../input/optiver-realized-volatility-prediction/book_train.parquet | wc -l

# Custom functions

In [ ]:
def log_return(list_stock_prices):
    return np.log(list_stock_prices).diff()

In [ ]:
def realized_volatility(series_log_return):
    return np.sqrt(np.sum(series_log_return**2))

# Explore data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

print(train_df.shape)
print(test_df.shape)
print(submit_df.shape)

In [ ]:
train_df.head(2)

In [ ]:
print(f"Unique stock ids: {train_df['stock_id'].nunique()}")
print(f"Unique time ids: {train_df['time_id'].nunique()}")

In [ ]:
# book_df = pd.read_parquet(f"/kaggle/input/optiver-realized-volatility-prediction/book_train.parquet/stock_id=0")

# print(book_df.shape)
# display(book_df.head())

In [ ]:
# book_df = book_df[book_df['seconds_in_bucket']>=500]

In [ ]:
# book_df['temp_bid_price_diff'] = (book_df['bid_price1'].diff())

In [ ]:
# trade_df = pd.read_parquet(f"/kaggle/input/optiver-realized-volatility-prediction/trade_train.parquet/stock_id=0")

# print(trade_df.shape)
# display(trade_df.head())

In [ ]:
# temp = opt.nru_features(0)
# temp1 = opt.ru_features(temp, stock=0)
# temp1.shape

In [ ]:
# temp1.isnull().sum().sum()

# Feature Engineering - Illidan

In [ ]:
# model_cols = ['real_vol1',
#  'real_vol4',
#  'real_vol2',
#  'OB_wap2_dollar_change_abs_sum',
#  'OB_wap4_dollar_change_abs_sum',
#  'real_vol3',
#  'OB_wap1_dollar_change_abs_sum',
#  'price_spread_sum',
#  'OB_wap3_dollar_change_abs_sum',
#  'bid_ask_diff_sum',
#  'spread_sum',
#  'volume_imbalance_min',
#  'stock_id',
#  'wapbal_sum',
#  'bid_ask_diff_wap2_ratio_mean',
#  'spread2_max',
#  'spread2_sum',
#  'bid_ask_diff2_wap2_ratio_mean',
#  'bid_ask_diff_wap_ratio_mean',
#  'price_spread2_sum',
#  'bid_spread_min',
#  'wap1_range']

# Order Book features

In [ ]:
# stockLst = train_df['stock_id'].unique().tolist()
# results = []

# for stock in tqdm(stockLst):
#     temp = opt.nru_features(stock)
#     results.append(opt.ru_features(temp, stock=stock))
    
#     del temp
#     _ = gc.collect()

In [ ]:
temp_df = pd.concat(results)
temp_df.shape

In [ ]:
temp_df = temp_df.reset_index(drop=False)
temp_df.index

In [ ]:
temp_df.columns

In [ ]:
temp_df.to_feather('train_rollup.feather')

In [ ]:
stockLst = train_df['stock_id'].unique().tolist()
results_500 = []
results_400 = []
results_300 = []
results_200 = []
results_100 = []

for stock in tqdm(stockLst):
    temp = opt.nru_features(stock)
    results_500.append(opt.ru_features(temp, stock=stock, since_second=500))
    results_400.append(opt.ru_features(temp, stock=stock, since_second=400))
    results_300.append(opt.ru_features(temp, stock=stock, since_second=300))
    results_200.append(opt.ru_features(temp, stock=stock, since_second=200))
    results_100.append(opt.ru_features(temp, stock=stock, since_second=100))
   
    del temp
    _ = gc.collect()

In [ ]:
temp500_df = pd.concat(results_500)
temp500_df = temp500_df.reset_index(drop=False)
temp500_df.to_feather('train_rollup_500.feather')

del temp500_df, results_500
_ = gc.collect()

In [ ]:
temp400_df = pd.concat(results_400)
temp400_df = temp400_df.reset_index(drop=False)
temp400_df.to_feather('train_rollup_400.feather')

del temp400_df, results_400
_ = gc.collect()

In [ ]:
temp300_df = pd.concat(results_300)
temp300_df = temp300_df.reset_index(drop=False)
temp300_df.to_feather('train_rollup_300.feather')

del temp300_df, results_300
_ = gc.collect()

In [ ]:
temp200_df = pd.concat(results_200)
temp200_df = temp200_df.reset_index(drop=False)
temp200_df.to_feather('train_rollup_200.feather')

del temp200_df, results_200
_ = gc.collect()

In [ ]:
temp100_df = pd.concat(results_100)
temp100_df = temp100_df.reset_index(drop=False)
temp100_df.to_feather('train_rollup_100.feather')

del temp100_df, results_100
_ = gc.collect()

In [ ]:
stockLst = train_df['stock_id'].unique().tolist()

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(opt.nru_features, stockLst), total=len(stockLst)))    

In [ ]:
import functools

stockLst = train_df['stock_id'].unique().tolist()

partial_order_book_features_500 = functools.partial(opt.order_book_features, since_second=500)
partial_order_book_features_400 = functools.partial(opt.order_book_features, since_second=400)
partial_order_book_features_300 = functools.partial(opt.order_book_features, since_second=300)
partial_order_book_features_200 = functools.partial(opt.order_book_features, since_second=200)
partial_order_book_features_100 = functools.partial(opt.order_book_features, since_second=100)


In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_500 = list(tqdm(executor.map(partial_order_book_features_500, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df = pd.concat(results)
opt_train_df_500 = pd.concat(results_500)

print(opt_train_df.shape)

del results, results_500
_ = gc.collect()

In [ ]:
opt_train_df_500.columns = [f'{col}_500' if col not in ['time_id','stock_id'] else col for col in opt_train_df_500.columns]

opt_train_df = opt.additional_feats(opt_train_df, suffix = "")
opt_train_df_500 = opt.additional_feats(opt_train_df_500, suffix = "_500")

opt_train_df = pd.merge(opt_train_df, opt_train_df_500, on=["time_id","stock_id"], how="left")

del opt_train_df_500
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_400 = list(tqdm(executor.map(partial_order_book_features_400, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_400 = pd.concat(results_400)

del results_400
_ = gc.collect()

opt_train_df_400.columns = [f'{col}_400' if col not in ['time_id','stock_id'] else col for col in opt_train_df_400.columns]

opt_train_df_400 = opt.additional_feats(opt_train_df_400, suffix = "_400")

opt_train_df = pd.merge(opt_train_df, opt_train_df_400, on=["time_id","stock_id"], how="left")

del opt_train_df_400
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_300 = list(tqdm(executor.map(partial_order_book_features_300, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_300 = pd.concat(results_300)

del results_300
_ = gc.collect()

opt_train_df_300.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_300.columns]

opt_train_df_300 = opt.additional_feats(opt_train_df_300, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_300, on=["time_id","stock_id"], how="left")

del opt_train_df_300
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_200 = list(tqdm(executor.map(partial_order_book_features_200, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_200 = pd.concat(results_200)

del results_200
_ = gc.collect()

opt_train_df_200.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_200.columns]

opt_train_df_200 = opt.additional_feats(opt_train_df_200, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_200, on=["time_id","stock_id"], how="left")

del opt_train_df_200
_ = gc.collect()

opt_train_df.shape

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    results_100 = list(tqdm(executor.map(partial_order_book_features_100, stockLst), total=len(stockLst)))

In [ ]:
opt_train_df_100 = pd.concat(results_100)

del results_100
_ = gc.collect()

opt_train_df_100.columns = [f'{col}_300' if col not in ['time_id','stock_id'] else col for col in opt_train_df_100.columns]

opt_train_df_100 = opt.additional_feats(opt_train_df_100, suffix = "_300")

opt_train_df = pd.merge(opt_train_df, opt_train_df_100, on=["time_id","stock_id"], how="left")

del opt_train_df_100
_ = gc.collect()

opt_train_df.shape

# Feature Engineering - Public 1

In [ ]:
train, test = opt2.read_train_test()

# data directory
data_dir = '../input/optiver-realized-volatility-prediction/'

In [ ]:
# Get unique stock ids 
train_stock_ids = train['stock_id'].unique()
# Preprocess them using Parallel and our single stock id functions
train_ = opt2.preprocessor(train_stock_ids, is_train = True)
train = train.merge(train_, on = ['row_id'], how = 'left')

# Get unique stock ids 
test_stock_ids = test['stock_id'].unique()
# Preprocess them using Parallel and our single stock id functions
test_ = opt2.preprocessor(test_stock_ids, is_train = False)
test = test.merge(test_, on = ['row_id'], how = 'left')
gc.collect()

In [ ]:
train['trade_size_tau'] = np.sqrt(1/train['trade_order_count_sum'])
    
for w in range(150, 600, 150):
    train['trade_size_tau_150win_'+str(w)] = np.sqrt(1/train['trade_order_count_sum_150win_'+str(w)])

In [ ]:
test['trade_size_tau'] = np.sqrt(1/test['trade_order_count_sum'])
    
for w in range(150, 600, 150):
    test['trade_size_tau_150win_'+str(w)] = np.sqrt(1/test['trade_order_count_sum_150win_'+str(w)])

In [ ]:
print(train.shape)

train.head()

In [ ]:
train = train.reset_index(drop=True)
train.to_feather("publicFeats1.feather")

In [ ]:
cols = ['row_id','time_id','stock_id','target']
cols+=['book_log_return1_realized_volatility']
cols+=['book_log_return1_realized_volatility_150win_150']+['book_log_return2_realized_volatility_150win_150']+[
            'trade_log_return_realized_volatility_150win_150']
cols+=['book_log_return1_realized_absvar_150win_150']+['book_log_return2_realized_absvar_150win_150']+[
            'trade_log_return_realized_absvar_150win_150']
cols+=['book_log_return2_realized_volatility_150win_300']
cols+=['book_log_return1_realized_volatility_150win_450']+['trade_log_return_realized_volatility_150win_450']
cols+=['book_price_spread_sum_150win_150']
cols+=['trade_size_tau_150win_150']
cols+=['book_depth_sum_150win_150']
cols+=['book_dispersion_sum_150win_150']

train_trunc = train[[col for col in train.columns if col in cols]]
test_trunc = test[[col for col in test.columns if col in cols]]

# Feature Engineering - Public 2

In [ ]:
path_submissions = '/'

target_name = 'target'
scores_folds = {}

# data directory
data_dir = '../input/optiver-realized-volatility-prediction/'

In [ ]:
# Read train and test
train =pd.read_pickle("/kaggle/input/optiver006/train.pkl")
test = opt3.read_train_test()

In [ ]:
# Get unique stock ids 
test_stock_ids = test['stock_id'].unique()
# Preprocess them using Parallel and our single stock id functions
test_ = opt3.preprocessor(test_stock_ids, is_train = False)
test = test.merge(test_, on = ['row_id'], how = 'left')

# Get group stats of time_id and stock_id
#train = get_time_stock(train)
test = opt3.get_time_stock(test)

train1=train
test1=test

In [ ]:
# replace by order sum (tau)
train['size_tau'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique'] )
test['size_tau'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique'] )
#train['size_tau_450'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_450'] )
#test['size_tau_450'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique_450'] )
train['size_tau_400'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_400'] )
test['size_tau_400'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique_400'] )
train['size_tau_300'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_300'] )
test['size_tau_300'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique_300'] )
#train['size_tau_150'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_150'] )
#test['size_tau_150'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique_150'] )
train['size_tau_200'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_200'] )
test['size_tau_200'] = np.sqrt( 1/ test['trade_seconds_in_bucket_count_unique_200'] )

In [ ]:
train['size_tau2'] = np.sqrt( 1/ train['trade_order_count_sum'] )
test['size_tau2'] = np.sqrt( 1/ test['trade_order_count_sum'] )
#train['size_tau2_450'] = np.sqrt( 0.25/ train['trade_order_count_sum'] )
#test['size_tau2_450'] = np.sqrt( 0.25/ test['trade_order_count_sum'] )
train['size_tau2_400'] = np.sqrt( 0.33/ train['trade_order_count_sum'] )
test['size_tau2_400'] = np.sqrt( 0.33/ test['trade_order_count_sum'] )
train['size_tau2_300'] = np.sqrt( 0.5/ train['trade_order_count_sum'] )
test['size_tau2_300'] = np.sqrt( 0.5/ test['trade_order_count_sum'] )
#train['size_tau2_150'] = np.sqrt( 0.75/ train['trade_order_count_sum'] )
#test['size_tau2_150'] = np.sqrt( 0.75/ test['trade_order_count_sum'] )
train['size_tau2_200'] = np.sqrt( 0.66/ train['trade_order_count_sum'] )
test['size_tau2_200'] = np.sqrt( 0.66/ test['trade_order_count_sum'] )

# delta tau
train['size_tau2_d'] = train['size_tau2_400'] - train['size_tau2']
test['size_tau2_d'] = test['size_tau2_400'] - test['size_tau2']

In [ ]:
colNames = [col for col in list(train.columns)
            if col not in {"stock_id", "time_id", "target", "row_id"}]
len(colNames)

In [ ]:
from sklearn.cluster import KMeans
# making agg features

train_p = pd.read_csv('../input/optiver-realized-volatility-prediction/train.csv')
train_p = train_p.pivot(index='time_id', columns='stock_id', values='target')

corr = train_p.corr()

ids = corr.index

kmeans = KMeans(n_clusters=7, random_state=0).fit(corr.values)
print(kmeans.labels_)

l = []
for n in range(7):
    l.append ( [ (x-1) for x in ( (ids+1)*(kmeans.labels_ == n)) if x > 0] )
    

mat = []
matTest = []

n = 0
for ind in l:
    print(ind)
    newDf = train.loc[train['stock_id'].isin(ind) ]
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    mat.append ( newDf )
    
    newDf = test.loc[test['stock_id'].isin(ind) ]    
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    matTest.append ( newDf )
    
    n+=1
    
mat1 = pd.concat(mat).reset_index()
mat1.drop(columns=['target'],inplace=True)

mat2 = pd.concat(matTest).reset_index()

In [ ]:
mat2 = pd.concat([mat2,mat1.loc[mat1.time_id==5]])
mat1 = mat1.pivot(index='time_id', columns='stock_id')
mat1.columns = ["_".join(x) for x in mat1.columns.ravel()]
mat1.reset_index(inplace=True)

mat2 = mat2.pivot(index='time_id', columns='stock_id')
mat2.columns = ["_".join(x) for x in mat2.columns.ravel()]
mat2.reset_index(inplace=True)

In [ ]:
nnn = ['time_id',
     'log_return1_realized_volatility_0c1',
     'log_return1_realized_volatility_1c1',     
     'log_return1_realized_volatility_3c1',
     'log_return1_realized_volatility_4c1',     
     'log_return1_realized_volatility_6c1',
     'total_volume_sum_0c1',
     'total_volume_sum_1c1', 
     'total_volume_sum_3c1',
     'total_volume_sum_4c1', 
     'total_volume_sum_6c1',
     'trade_size_sum_0c1',
     'trade_size_sum_1c1', 
     'trade_size_sum_3c1',
     'trade_size_sum_4c1', 
     'trade_size_sum_6c1',
     'trade_order_count_sum_0c1',
     'trade_order_count_sum_1c1',
     'trade_order_count_sum_3c1',
     'trade_order_count_sum_4c1',
     'trade_order_count_sum_6c1',      
     'price_spread_sum_0c1',
     'price_spread_sum_1c1',
     'price_spread_sum_3c1',
     'price_spread_sum_4c1',
     'price_spread_sum_6c1',   
     'bid_spread_sum_0c1',
     'bid_spread_sum_1c1',
     'bid_spread_sum_3c1',
     'bid_spread_sum_4c1',
     'bid_spread_sum_6c1',       
     'ask_spread_sum_0c1',
     'ask_spread_sum_1c1',
     'ask_spread_sum_3c1',
     'ask_spread_sum_4c1',
     'ask_spread_sum_6c1',   
     'volume_imbalance_sum_0c1',
     'volume_imbalance_sum_1c1',
     'volume_imbalance_sum_3c1',
     'volume_imbalance_sum_4c1',
     'volume_imbalance_sum_6c1',       
     'bid_ask_spread_sum_0c1',
     'bid_ask_spread_sum_1c1',
     'bid_ask_spread_sum_3c1',
     'bid_ask_spread_sum_4c1',
     'bid_ask_spread_sum_6c1',
     'size_tau2_0c1',
     'size_tau2_1c1',
     'size_tau2_3c1',
     'size_tau2_4c1',
     'size_tau2_6c1'] 
train = pd.merge(train,mat1[nnn],how='left',on='time_id')
test = pd.merge(test,mat2[nnn],how='left',on='time_id')

In [ ]:
import gc
del mat1,mat2
gc.collect()

In [ ]:
# add extra features based on highest feature importance

train['total_volume_sum_0c1/trade_size_sum_0c1'] = train['total_volume_sum_0c1'] / train['trade_size_sum_0c1']
test['total_volume_sum_0c1/trade_size_sum_0c1'] = test['total_volume_sum_0c1'] / test['trade_size_sum_0c1']

train['total_volume_sum_3c1/size_tau2_3c1'] = train['total_volume_sum_3c1'] / train['size_tau2_3c1']
test['total_volume_sum_3c1/size_tau2_3c1'] = test['total_volume_sum_3c1'] / test['size_tau2_3c1']

train['total_volume_sum_1c1/size_tau2_1c1'] = train['total_volume_sum_1c1'] / train['size_tau2_1c1']
test['total_volume_sum_1c1/size_tau2_1c1'] = test['total_volume_sum_1c1'] / test['size_tau2_1c1']

train['stock_id/price_spread_sum'] = train['stock_id'] / train['price_spread_sum']
test['stock_id/price_spread_sum'] = test['stock_id'] / test['price_spread_sum']

In [ ]:
train.shape

In [ ]:
train = train.reset_index(drop=True)
train.to_feather("publicFeats2.feather")

In [ ]:
!ls -hlt

# Recreate metrics from starter notebook

https://www.kaggle.com/jiashenliu/introduction-to-financial-concepts-and-data#Naive-prediction:-using-past-realized-volatility-as-target

In [ ]:
opt_train_df = pd.merge(opt_train_df, train_df, on=['stock_id','time_id'], how="left")

opt_train_df.shape

In [ ]:
from sklearn.metrics import r2_score
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))
R2 = round(r2_score(y_true = opt_train_df['target'], y_pred = opt_train_df['real_vol1']),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'], y_pred = opt_train_df['real_vol1']),3)
print(f'Performance of the naive prediction: R2 score: {R2}, RMSPE: {RMSPE}')

# Save file

In [ ]:
opt_train_df.to_feather("optiver_train.feather")

In [ ]:
!ls -hlt /kaggle/working/